### **📦 Imports and Setup**

In [56]:
import pandas as pd
import os, glob, json, re

### **📑 Load Station Codes**

In [57]:
with open("json/station-code/station-code.json", "r", encoding="utf-8") as f:
    station_list = json.load(f)

# Convert list -> dict for quick lookup
station_map = {item["code"].upper(): item["name"] for item in station_list}

# Ensure output folder exists
os.makedirs("json", exist_ok=True)

### **🔧 Helper Functions**

##### **1. "CLEAN_RELASI" Processing**

In [58]:
def make_clean_relasi_from_relasi(relasi_str, keterangan_type):
    """Build CLEAN_RELASI from RELASI + KETERANGAN type logic."""
    if not isinstance(relasi_str, str):
        return ""

    # Normalize codes
    codes = [c.strip().upper() for c in relasi_str.split("-") if c.strip()]
    if not codes:
        return ""

    start, end = codes[0], codes[-1]
    start_name = station_map.get(start, start)
    end_name = station_map.get(end, end)

    # ----------------------
    # CLEAN_RELASI rules
    # ----------------------
    if len(codes) == 2:
        return f"{start_name} - {end_name}"

    elif len(codes) == 3:
        # Add the middle one as VIA
        via = station_map.get(codes[1], codes[1])
        return f"{start_name} - {end_name} via {via}"

    else:
        # More than 3 stations (non-Full Racket) → keep only start-end
        # (avoids crazy long strings like "via A-B-C")
        middle_stations = [station_map.get(c, c) for c in codes[1:-1]]
        via_str = " & ".join(middle_stations)
        return f"{start_name} - {end_name} via {via_str}"

##### **2. "KETERANGAN" Processing**

In [59]:
def parse_keterangan(row, next_row):
    """Return structured KETERANGAN info."""
    raw = str(row["KETERANGAN"]).strip()
    norm_raw = normalize_whitespace(raw)   # normalize whitespace

    # Default structure
    keterangan_obj = {
        "text": norm_raw,              
        "type": "None",
        "note": "",
        "next_nomor_ka": ""
    }

    # ---------------------------
    # --- Case 1: Full Racket ---
    # ---------------------------
    if "Full Racket" in norm_raw:
        keterangan_obj["type"] = "Full Racket"
        keterangan_obj["keterangan"] = "Full Racket"

        if " - " in str(row["NOMOR_KA"]) and " - " in str(row["RELASI"]):
            try:
                parts_nomor = [p.strip() for p in str(row["NOMOR_KA"]).split(" - ")]
                parts_relasi = [p.strip() for p in str(row["RELASI"]).split(" - ")]

                if len(parts_nomor) == 2 and len(parts_relasi) == 2:
                    row["NOMOR_KA"] = parts_nomor[0]
                    row["RELASI"] = parts_relasi[0]
                    keterangan_obj["next_nomor_ka"] = parts_nomor[1]
            except Exception:
                pass
        elif next_row is not None:
            keterangan_obj["next_nomor_ka"] = next_row.get("NOMOR_KA", "")

    # ------------------------------------
    # --- Case 2: Info (special notes) ---
    # ------------------------------------
    elif "Sabtu/Minggu/Libur Nasional Batal" in norm_raw:
        keterangan_obj["type"] = "Holiday"
        keterangan_obj["note"] = "Batal saat Sabtu/Minggu/Libur Nasional"

    return keterangan_obj

##### **3. Normalize Whitespace**

In [60]:
def normalize_whitespace(s: str) -> str:
    """Replace multiple whitespace (space, tab, etc.) with a single space."""
    return re.sub(r"\s+", " ", s).strip()

##### **4. Build Stop Station**

In [61]:
def assign_stops(row: dict, station_map: dict) -> list:
    """
    Build stops array directly from row keys (keeps original column order).
    
    Args:
        row (dict): Row dict from dataframe
        station_map (dict): Mapping { "MRI": "Manggarai", ... }

    Returns:
        list: Stops array [{code, name, time}, ...]
    """
    stops = []
    for code, value in row.items():
        if code in ["NO", "NOMOR_KA", "RELASI", "CLEAN_RELASI", "KETERANGAN", "STOPS"]:
            continue  # skip metadata columns

        if isinstance(value, dict):
            continue  # skip nested dicts like KETERANGAN

        # normalize time
        time = "" if value in ["", "Ls"] else value

        stops.append({
            "code": code,
            "name": station_map.get(code, code),
            "time": time
        })
    return stops

### **📂 Process All CSV Files**

In [62]:
for file in glob.glob("csv/*.csv"):
    print(f"Processing: {file}")

    # --------------------
    # --- Load raw CSV ---
    # --------------------
    df = pd.read_csv(file, header=None)

    # -------------------------
    # --- Clean header rows ---
    # -------------------------
    df = df.drop(0).reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.drop(0).reset_index(drop=True)

    # Extract station row
    station_row = df.iloc[0].fillna("")
    df = df.drop(0).reset_index(drop=True)

    base_cols = ["NO", "NOMOR_KA", "RELASI"]
    station_cols = station_row.tolist()[3:-1]
    station_cols = [c if c != "" else f"STN{i}" for i, c in enumerate(station_cols, 1)]
    final_cols = base_cols + station_cols + ["KETERANGAN"]

    # Ensure column count matches
    if len(final_cols) == df.shape[1]:
        df.columns = final_cols
    else:
        print(f"Header mismatch in {file}")
        continue

    df = df.fillna("")

    # ------------------------
    # --- Process each row ---
    # ------------------------
    rows = []
    skip_next = False
    for i in range(len(df)):
        if skip_next:
            skip_next = False
            continue

        row = df.iloc[i].to_dict()
        next_row = df.iloc[i + 1].to_dict() if i + 1 < len(df) else None

        # -----------------------------
        # --- Keterangan Processing ---
        # -----------------------------
        keterangan_obj = parse_keterangan(row, next_row)
        # If Full Racket but not duplication → skip next row
        if keterangan_obj["type"] == "Full Racket" and next_row is not None and keterangan_obj["next_nomor_ka"] == next_row["NOMOR_KA"]:
            skip_next = True

        # -------------------------------
        # --- CLEAN_RELASI Processing ---
        # -------------------------------
        row["CLEAN_RELASI"] = make_clean_relasi_from_relasi(row["RELASI"], keterangan_obj["type"])
        row["KETERANGAN"] = keterangan_obj

        # -------------------------
        # --- STOPS Processing ---
        # -------------------------
        row["STOPS"] = assign_stops(row, station_map)

        # clean columns
        clean_row = {
            "NO": row["NO"],
            "NOMOR_KA": row["NOMOR_KA"],
            "RELASI": row["RELASI"],
            "CLEAN_RELASI": row["CLEAN_RELASI"],
            "KETERANGAN": row["KETERANGAN"],
            "STOPS": row["STOPS"]
        }

        rows.append(clean_row)

    # ------------------------
    # --- Save output JSON ---
    # ------------------------
    out_df = pd.DataFrame(rows)
    base_name = os.path.splitext(os.path.basename(file))[0]
    out_file = f"json/krl-schedule/{base_name}.json"
    out_df.to_json(out_file, orient="records", force_ascii=False, indent=2)
    print(f"Saved {out_file}\n\n")

Processing: csv\krl-schedule - from-bogor.csv
Saved json/krl-schedule/krl-schedule - from-bogor.json


Processing: csv\krl-schedule - from-cikarang.csv
Saved json/krl-schedule/krl-schedule - from-cikarang.json


Processing: csv\krl-schedule - from-rangkasbitung.csv
Saved json/krl-schedule/krl-schedule - from-rangkasbitung.json


Processing: csv\krl-schedule - from-soekarnohatta.csv
Saved json/krl-schedule/krl-schedule - from-soekarnohatta.json


Processing: csv\krl-schedule - from-tangerang.csv
Saved json/krl-schedule/krl-schedule - from-tangerang.json


Processing: csv\krl-schedule - from-tanjungpriok.csv
Saved json/krl-schedule/krl-schedule - from-tanjungpriok.json


Processing: csv\krl-schedule - to-bogor.csv
Saved json/krl-schedule/krl-schedule - to-bogor.json


Processing: csv\krl-schedule - to-cikarang.csv
Saved json/krl-schedule/krl-schedule - to-cikarang.json


Processing: csv\krl-schedule - to-rangkasbitung.csv
Saved json/krl-schedule/krl-schedule - to-rangkasbitung.json


Pro

### **Testing with Table Interface**

In [63]:
import os
import glob
import pandas as pd

# Path to folder
folder = "json/krl-schedule/"

# Loop over all JSON files
for file in glob.glob(os.path.join(folder, "*.json")):
    print(f"\n=== Preview: {os.path.basename(file)} ===")
    print(f"First 10 Rows:")

    try:
        # Load JSON
        df = pd.read_json(file)

        # Flatten KETERANGAN if exists
        if "KETERANGAN" in df.columns:
            keterangan_df = pd.json_normalize(df["KETERANGAN"])
            keterangan_df = keterangan_df.add_prefix("KETERANGAN.")
            df = df.drop(columns=["KETERANGAN"]).join(keterangan_df)

        # Flatten STOPS if exists
        if "STOPS" in df.columns:
            stops_df = df["STOPS"].apply(
                lambda stops: {s["name"]: s["time"] for s in stops}
            )
            stops_df = pd.DataFrame(stops_df.tolist())
            df = df.drop(columns=["STOPS"]).join(stops_df)

        # Show first 10 rows
        print(df.head(10).to_markdown(index=False), "\n\n")

    except Exception as e:
        print(f"Error reading {file}: {e}")


=== Preview: krl-schedule - from-bogor.json ===
First 10 Rows:
|   NO | NOMOR_KA   | RELASI   | CLEAN_RELASI         | KETERANGAN.text                   | KETERANGAN.type   | KETERANGAN.note                        | KETERANGAN.next_nomor_ka   | Nambo   | Cibinong   | Pondok Rajeg   | Bogor   | Cilebut   | Bojong Gede   | Citayam   | Depok   | Depok Baru   | Pondok Cina   | Univ. Indonesia   | Univ. Pancasila   | Lenteng Agung   | Tanjung Barat   | Pasar Minggu   | Pasar Minggu Baru   | Duren Kalibata   | Cawang   | Tebet   | Manggarai   | Cikini   | Gondangdia   | Gambir   | Juanda   | Sawah Besar   | Mangga Besar   | Jayakarta   | Jakarta Kota   |
|-----:|:-----------|:---------|:---------------------|:----------------------------------|:------------------|:---------------------------------------|:---------------------------|:--------|:-----------|:---------------|:--------|:----------|:--------------|:----------|:--------|:-------------|:--------------|:------------------|:---------